# 00_env_config

Environment bootstrap for FabricOps Starter Kit notebooks.
This notebook defines environment-wide values and assembles framework config.
Reusable functions come from `fabricops_kit` package modules.


In [ ]:
from fabricops_kit.fabric_input_output import (
    FabricStore,
    check_naming_convention,
    read_lakehouse_csv,
    read_lakehouse_table,
    write_lakehouse_table,
    read_warehouse_table,
    write_warehouse_table,
)
from fabricops_kit.config import (
    AIPromptConfig,
    FrameworkConfig,
    GovernanceConfig,
    LineageConfig,
    NotebookRuntimeConfig,
    PathConfig,
    QualityConfig,
    ReviewWorkflowConfig,
    load_config,
    setup_notebook,
)


In [ ]:
ENV = "dev"
VALIDATION_MODE = "warn"
NOTEBOOK_PREFIXES = ("00_env_config", "01_dsa_", "02_ex_", "03_pc_")


In [ ]:
ENV_PATHS = {
    "dev": {
        "source": FabricStore(env="dev", workspace_id="00000000-0000-0000-0000-000000000001", item_id="11111111-1111-1111-1111-111111111111", name="lh_source_dev", kind="lakehouse"),
        "unified": FabricStore(env="dev", workspace_id="00000000-0000-0000-0000-000000000001", item_id="22222222-2222-2222-2222-222222222222", name="lh_unified_dev", kind="lakehouse"),
        "product": FabricStore(env="dev", workspace_id="00000000-0000-0000-0000-000000000001", item_id="33333333-3333-3333-3333-333333333333", name="wh_product_dev", kind="warehouse"),
        "metadata": FabricStore(env="dev", workspace_id="00000000-0000-0000-0000-000000000001", item_id="44444444-4444-4444-4444-444444444444", name="lh_metadata_dev", kind="lakehouse"),
    }
}


## AI prompt overrides


### Business context prompt


In [ ]:
BUSINESS_CONTEXT_PROMPT_TEMPLATE = """
You are helping draft the business context for a FabricOps data sharing agreement notebook.

This output is advisory only.
A human data owner, steward, analyst, or engineer must review and approve it before it is treated as official agreement context.

Your job is to convert agreement metadata, source descriptions, notebook notes, and available evidence into clear business context that downstream exploration and pipeline contract notebooks can use.

Focus on:
1. What business question or operational need this data supports.
2. Who owns or stewards the data.
3. Who is allowed to use the data.
4. What the data can be used for.
5. What the data must not be used for.
6. Any sensitivity, privacy, or governance concerns.
7. Any known limitations, caveats, or open questions.

Do not invent facts.
If information is missing, mark it as an open question.
Do not approve access, usage, classifications, or data quality rules.
Do not write legal language.
Use plain business language.

Return only a Python dictionary named BUSINESS_CONTEXT using this shape:

BUSINESS_CONTEXT = {
    "agreement_id": "{agreement_id}",
    "agreement_name": "{agreement_name}",
    "business_purpose": "Plain explanation of why this data is needed.",
    "primary_users": ["team_or_role"],
    "data_owner": "Known owner or 'unknown'.",
    "data_steward": "Known steward or 'unknown'.",
    "approved_usage": ["usage"],
    "restricted_usage": ["restriction"],
    "key_business_terms": {"term": "definition"},
    "known_limitations": ["limitation"],
    "governance_notes": ["note"],
    "open_questions": ["question"]
}

Agreement metadata:
{agreement_metadata}

Source evidence:
{source_evidence}

User notes:
{user_notes}
"""


### DQ rule suggestion prompt


In [ ]:
DQ_RULE_SUGGESTION_PROMPT_TEMPLATE = """
You are helping draft candidate FabricOps data quality rules for a pipeline contract notebook.

These suggestions are advisory only.
A human analyst or engineer must approve them before enforcement.

Use only these FabricOps rule_type values:

1. not_null
   Use when a column must be populated.
   Required fields:
   rule_id, rule_type, columns, severity, description

2. unique_key
   Use when one or more columns define the business grain and must be unique.
   Required fields:
   rule_id, rule_type, columns, severity, description

3. accepted_values
   Use when a column should only contain known business values.
   Required fields:
   rule_id, rule_type, columns, allowed_values, severity, description

4. value_range
   Use when a numeric, date, or timestamp column should stay within a sensible range.
   Required fields:
   rule_id, rule_type, columns, lower_bound or upper_bound, severity, description

5. regex_format
   Use when a string column should match a known format such as email, code, phone, postal code, or ID.
   Required fields:
   rule_id, rule_type, columns, regex_pattern, severity, description

Heuristics:
- Suggest not_null when null_count is 0 or when the column name looks mandatory, such as id, key, date, code, status, amount, or name.
- Suggest unique_key only when distinct_count is close to row_count and the column name looks like an identifier or business key.
- Suggest accepted_values when distinct_count is small and the observed values look like business categories.
- Suggest value_range only when lower_bound and upper_bound are available and the range is business meaningful.
- Suggest regex_format only for clear format columns such as email, phone, postal_code, programme_code, course_code, invoice_number, or staff_id.
- Use severity="error" only for rules that should block the pipeline.
- Use severity="warning" for rules that should be reviewed but should not block the pipeline.
- Do not suggest unsupported rule types.
- Do not return Great Expectations, Deequ, DQX, SQL, or pseudocode syntax.

Return only a Python dictionary named DQ_RULES using this shape:

DQ_RULES = {
    "{table_name}": [
        {
            "rule_id": "lower_snake_case_rule_id",
            "rule_type": "one_supported_rule_type",
            "columns": ["column_name"],
            "severity": "error_or_warning",
            "description": "Plain business explanation."
        }
    ]
}

For accepted_values, include allowed_values.
For value_range, include lower_bound and/or upper_bound.
For regex_format, include regex_pattern.

Table name:
{table_name}

Column profile row:
Column name: {column_name}
Data type: {data_type}
Row count: {row_count}
Null count: {null_count}
Null percent: {null_percent}
Distinct count: {distinct_count}
Distinct percent: {distinct_percent}
Minimum value: {min_value}
Maximum value: {max_value}
Observed values sample: {observed_values_sample}

Approved business context:
{approved_business_context}
""".strip()


### Governance personal identifier prompt


In [ ]:
GOVERNANCE_PERSONAL_IDENTIFIER_PROMPT_TEMPLATE = """
You are helping identify potential personal identifiers in a FabricOps data sharing agreement or exploration notebook.

This output is advisory only.
A human data steward or governance reviewer must approve the final classification.

Your job is to review column names, descriptions, data types, samples, and business context to identify columns that may directly or indirectly identify a person.

Classify each candidate as one of: direct_identifier, quasi_identifier, sensitive_personal_attribute, not_personal_identifier.

Return only a Python dictionary named PERSONAL_IDENTIFIER_CANDIDATES using this shape:
PERSONAL_IDENTIFIER_CANDIDATES = {"{table_name}": [{"column": "column_name", "classification": "direct_identifier | quasi_identifier | sensitive_personal_attribute | not_personal_identifier", "confidence": "high | medium | low", "reason": "Plain explanation.", "recommended_review_action": "approve | review | ignore"}]}

Table name:
{table_name}

Business context:
{business_context}

Column evidence:
Column name: {column_name}
Data type: {data_type}
Description: {column_description}
Observed values sample: {observed_values_sample}
"""


### Governance candidate classification prompt


In [ ]:
GOVERNANCE_CANDIDATE_PROMPT_TEMPLATE = """
You are helping draft candidate governance classifications for a FabricOps data sharing agreement or pipeline contract.

These classifications are advisory only.
A human data steward, data owner, or governance reviewer must approve them before they are used for enforcement, access review, masking, or publication.

Return only a Python dictionary named GOVERNANCE_CANDIDATES using this shape:
GOVERNANCE_CANDIDATES = {"dataset": {"table_name": "{table_name}", "candidate_sensitivity": "public | internal | confidential | restricted", "candidate_privacy_label": "no_personal_data | personal_data | sensitive_personal_data", "confidence": "high | medium | low", "reason": "Plain explanation.", "open_questions": ["question"]}, "columns": [{"column": "column_name", "candidate_sensitivity": "public | internal | confidential | restricted", "candidate_privacy_label": "no_personal_data | personal_data | sensitive_personal_data", "confidence": "high | medium | low", "reason": "Plain explanation.", "recommended_controls": ["access_control | masking | aggregation | review_only"]}]}

Table name:
{table_name}
Business context:
{business_context}
Agreement restrictions:
{agreement_restrictions}
Personal identifier candidates:
{personal_identifier_candidates}
Column profile:
{column_profile}
"""


### Governance review prompt


In [ ]:
GOVERNANCE_REVIEW_PROMPT_TEMPLATE = """
You are helping prepare governance candidates for human review in a FabricOps workflow.

Return only a Python dictionary named GOVERNANCE_REVIEW using this shape:
GOVERNANCE_REVIEW = {"table_name": "{table_name}", "review_summary": "Concise summary.", "recommended_for_approval": [{"scope": "dataset | column", "name": "dataset_or_column_name", "recommendation": "Plain recommendation.", "reason": "Reason."}], "needs_human_review": [{"scope": "dataset | column", "name": "dataset_or_column_name", "issue": "What needs review.", "suggested_question": "Question to ask."}], "inconsistencies": [{"scope": "dataset | column", "name": "dataset_or_column_name", "issue": "Conflict or unsupported claim."}], "open_questions": ["question"]}

Table name:
{table_name}
Business context:
{business_context}
Governance candidates:
{governance_candidates}
Personal identifier candidates:
{personal_identifier_candidates}
Agreement restrictions:
{agreement_restrictions}
"""


### Handover summary prompt


In [ ]:
HANDOVER_SUMMARY_PROMPT_TEMPLATE = """
You are helping generate a FabricOps handover summary for a data product.

Return only a Python dictionary named HANDOVER_SUMMARY using this shape:
HANDOVER_SUMMARY = {"data_product_name": "{data_product_name}", "purpose": "Plain explanation.", "source_data": [{"source_name": "source", "description": "description"}], "approved_usage": ["usage"], "restricted_usage": ["restriction"], "transformations": ["transformation"], "outputs": [{"name": "output_name", "type": "lakehouse_table | warehouse_table | file | report | other", "description": "description"}], "data_quality": {"rules_applied": ["rule summary"], "known_gaps": ["gap"]}, "governance": {"sensitivity": "public | internal | confidential | restricted | unknown", "privacy_label": "no_personal_data | personal_data | sensitive_personal_data | unknown", "approved_by": "name_or_unknown", "notes": ["note"]}, "lineage": ["lineage step"], "operational_notes": ["note"], "known_risks": ["risk"], "open_questions": ["question"], "recommended_next_actions": ["action"]}

Data product name:
{data_product_name}
Agreement context:
{agreement_context}
Business context:
{business_context}
Source profile evidence:
{source_profile_evidence}
Transformation summary:
{transformation_summary}
Data quality evidence:
{data_quality_evidence}
Governance evidence:
{governance_evidence}
Lineage evidence:
{lineage_evidence}
Operational notes:
{operational_notes}
"""


In [ ]:
PATH_CONFIG = PathConfig(paths=ENV_PATHS)
RUNTIME_CONFIG = NotebookRuntimeConfig(
    allowed_notebook_prefixes=NOTEBOOK_PREFIXES,
)
AI_PROMPTS = AIPromptConfig(
    business_context_prompt_template=BUSINESS_CONTEXT_PROMPT_TEMPLATE,
    dq_rule_suggestion_prompt_template=DQ_RULE_SUGGESTION_PROMPT_TEMPLATE,
    governance_personal_identifier_prompt_template=GOVERNANCE_PERSONAL_IDENTIFIER_PROMPT_TEMPLATE,
    governance_candidate_prompt_template=GOVERNANCE_CANDIDATE_PROMPT_TEMPLATE,
    governance_review_prompt_template=GOVERNANCE_REVIEW_PROMPT_TEMPLATE,
    handover_summary_prompt_template=HANDOVER_SUMMARY_PROMPT_TEMPLATE,
)
QUALITY_CONFIG = QualityConfig()
GOVERNANCE_CONFIG = GovernanceConfig()
REVIEW_WORKFLOW_CONFIG = ReviewWorkflowConfig()
LINEAGE_CONFIG = LineageConfig()

CONFIG = FrameworkConfig(
    path_config=PATH_CONFIG,
    notebook_runtime_config=RUNTIME_CONFIG,
    ai_prompt_config=AI_PROMPTS,
    quality_config=QUALITY_CONFIG,
    governance_config=GOVERNANCE_CONFIG,
    review_workflow_config=REVIEW_WORKFLOW_CONFIG,
    lineage_config=LINEAGE_CONFIG,
)


In [ ]:
CONFIG = load_config(CONFIG)
BOOTSTRAP = setup_notebook(
    config=CONFIG,
    env=ENV,
    required_targets=["source", "unified", "product", "metadata"],
    notebook_name="00_env_config",
)
check_naming_convention(
    allowed_prefixes=NOTEBOOK_PREFIXES,
    fail_on_error=(VALIDATION_MODE == "strict"),
)


In [ ]:
print("FabricOps environment bootstrap ready")
print(f"- env: {ENV}")
print(f"- validation mode: {VALIDATION_MODE}")
print(f"- source target: {CONFIG.path_config.paths[ENV]['source'].name}")
print(f"- metadata target: {CONFIG.path_config.paths[ENV]['metadata'].name}")
